# AI Symptom Intake and Triage Support Website Generator

**Task 1 entry point:** This is the single Jupyter Notebook for the software component.

**Project concept:** A notebook-driven AI software generation workflow that creates and verifies a Flask + SQLite website for symptom intake, AI follow-up questions, structured summaries, non-diagnostic care guidance, profile-aware history use, checkup-text structuring, tests, UML, and SDLC documentation.

**Safety boundary:** The generated website is not a diagnosis tool, does not replace clinicians, and does not provide prescriptions. It uses red-flag checks, consent controls, backend-only AI calls, structured output validation, and fallback responses.

**Output folder:** `generated_project/`

**Purpose:** Demonstrate meta-software development: the notebook is the generator, and the Flask website is the generated artefact.


## 1. Title and Project Overview

**Purpose:** State the business problem and define the generated system.

**Output Files:** Notebook introduction, `generated_project/README.md`


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys
from textwrap import dedent

PROJECT = Path("generated_project")
DOCS = PROJECT / "docs"
TEMPLATES = PROJECT / "templates"
STATIC = PROJECT / "static"
TESTS = PROJECT / "tests"
UML_PNG = DOCS / "uml_png"

for folder in [PROJECT, DOCS, TEMPLATES, STATIC, TESTS, UML_PNG]:
    folder.mkdir(parents=True, exist_ok=True)

project_overview = {
    "title": "AI Symptom Intake and Triage Support Website",
    "generated_stack": ["Flask", "SQLite", "HTML/CSS/JS", "APIFree/OpenAI-compatible API", "pytest"],
    "main_outputs": ["Flask API", "website pages", "generated image", "SDLC documents", "UML", "tests"],
    "safety_scope": "symptom intake and triage support only; no diagnosis or prescription",
}
project_overview


## 2. Problem Definition

**Purpose:** Convert the business problem into a precise software generation target.

**Output Files:** `docs/requirements.md`


In [ ]:
business_problem = dedent('''
Users may feel unwell but may not know how to describe symptoms clearly before seeking care.
The generated system should collect symptoms conversationally, ask useful follow-up questions,
summarise the information, provide non-diagnostic next-step guidance, and help users locate
non-diagnostic care guidance.
''').strip()

stakeholders = ["Patient/User", "Student/Developer", "AI service", "Clinician/Pharmacist for final review"]
core_user_goals = [
    "Start a quick symptom intake",
    "Optionally use a saved health profile and history with consent",
    "Answer AI-generated follow-up questions",
    "Receive a structured symptom summary",
    "Review non-diagnostic care guidance",
]

print(business_problem)
print("Stakeholders:", stakeholders)


## 3. Objectives and Scope

**Purpose:** Define what the generated system does and what it intentionally avoids.

**Output Files:** `docs/safety_policy.md`, `docs/architecture_decisions.md`


In [ ]:
in_scope = [
    "symptom intake",
    "one-question-at-a-time AI follow-up",
    "structured summary",
    "non-diagnostic care guidance",
    "saved health profile with user consent",
    "manual checkup text structuring",
    "tests and CI/CD evidence",
]

out_of_scope = [
    "diagnosis",
    "prescription",
    "exact medication dosage",
    "emergency decision-making",
    "real patient record processing",
]

print("In scope:")
for item in in_scope:
    print("-", item)
print("\nOut of scope:")
for item in out_of_scope:
    print("-", item)


## 4. APIFree Setup

**Purpose:** Configure the OpenAI-compatible model layer used by the generated Flask backend and notebook generator.

**Security rule:** The real APIFree key is loaded from `.env` or environment variables only. The Notebook records the model configuration and whether a key is present, but it never prints or stores the secret value.

**Output Files:** `.env.example` is generated as a safe template. The real `.env` file is local-only and must not be committed.


In [ ]:
try:
    import requests
except ImportError:
    requests = None

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

ENV_FILE = PROJECT / ".env"
ENV_EXAMPLE = PROJECT / ".env.example"

ENV_EXAMPLE.write_text(dedent('''APIFREE_API_KEY=replace_with_your_apifree_key
APIFREE_BASE_URL=https://api.apifree.ai/v1
APIFREE_MODEL=deepseek-ai/deepseek-v4-flash
FLASK_ENV=development
'''), encoding="utf-8")

if load_dotenv and ENV_FILE.exists():
    load_dotenv(ENV_FILE)

APIFREE_API_KEY = os.getenv("APIFREE_API_KEY") or os.getenv("API_KEY")
APIFREE_BASE_URL = os.getenv("APIFREE_BASE_URL", "https://api.apifree.ai/v1")
APIFREE_MODEL = os.getenv("APIFREE_MODEL", "deepseek-ai/deepseek-v4-flash")

# Live regeneration is off by default so the notebook is safe to run without spending API credits.
ENABLE_LIVE_REGENERATION = os.getenv("ENABLE_LIVE_REGENERATION", "0") == "1"

ai_runtime_config = {
    "provider": "APIFree / OpenAI-compatible chat completions",
    "base_url": APIFREE_BASE_URL,
    "model": APIFREE_MODEL,
    "api_key_loaded": bool(APIFREE_API_KEY),
    "api_key_source": ".env or environment variable" if APIFREE_API_KEY else "not configured",
    "live_regeneration_enabled": ENABLE_LIVE_REGENERATION,
    "secret_printed": False,
}

assert not ENV_EXAMPLE.read_text(encoding="utf-8").startswith("APIFREE_API_KEY=sk-"), "Do not store a real key in .env.example"
ai_runtime_config


## 5. Prompt Engineering Strategy

**Purpose:** Define file-level prompts, JSON-only runtime prompts, and validation rules.

**Output Files:** `docs/prompt_design.md`


In [ ]:
BASE_PROMPT = dedent('''
You are generating one file for a coursework-grade Flask + SQLite AI symptom intake website.
Safety boundary: symptom intake and triage support only; no diagnosis, no prescriptions,
no exact medication dose, and no real patient data.
Technical constraints: Flask, SQLite, pytest, plain HTML/CSS/JS, backend-only APIFree calls,
no secrets in frontend.
Assessment constraints: one notebook in Task 1, generated website, generated image,
SDLC documentation, UML, tests, CI/CD and deployment evidence.
Return only the requested file content.
''').strip()

runtime_prompt_rules = {
    "followup": "Ask one short follow-up question at a time; return valid JSON.",
    "summary": "Summarise user input into structured non-diagnostic JSON.",
    "care_guidance": "Provide general next steps, not diagnosis or prescription.",
    "checkup_structure": "Convert pasted checkup text into structured FHIR-style fields.",
}

validation_rules = [
    "Backend validates JSON before AI calls",
    "Red-flag rules run before normal model guidance",
    "APIFree key stays server-side",
    "Fallback response is used if the model is unavailable or returns invalid JSON",
]

runtime_prompt_rules, validation_rules


## 6. Helper Functions for Generation

**Purpose:** Centralise file writing, model calls, fallback behaviour, and output validation.

The notebook uses a safe fallback pattern: if live APIFree regeneration is disabled or unavailable, it preserves the already generated coursework artefacts. This makes the submitted project reproducible without forcing the marker to spend API credits.

**Output Files:** Helper functions used by documentation, UML, backend, frontend, image prompt and test generation sections.


In [ ]:
def strip_code_fence(text: str) -> str:
    text = str(text or "").strip()
    match = re.match(r"^```[a-zA-Z0-9_-]*\n(.*)\n```$", text, flags=re.DOTALL)
    return match.group(1).strip() if match else text


def existing_text(path: Path, fallback: str) -> str:
    return path.read_text(encoding="utf-8") if path.exists() else fallback


def call_apifree(prompt: str, fallback: str) -> str:
    """Generate content with APIFree when explicitly enabled; otherwise reuse reviewed fallback."""
    if not (ENABLE_LIVE_REGENERATION and APIFREE_API_KEY and requests):
        return fallback

    response = requests.post(
        f"{APIFREE_BASE_URL.rstrip('/')}/chat/completions",
        headers={
            "Authorization": f"Bearer {APIFREE_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": APIFREE_MODEL,
            "messages": [
                {"role": "system", "content": "Return only the requested file content."},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0.2,
            "stream": False,
            "max_tokens": 8192,
        },
        timeout=90,
    )
    data = response.json()
    if "error" in data:
        raise RuntimeError(data["error"].get("message", "APIFree returned an error"))
    return strip_code_fence(data["choices"][0]["message"]["content"])


def write_generated_file(path: Path, prompt: str, fallback: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    content = call_apifree(prompt, fallback)
    path.write_text(content.rstrip() + "\n", encoding="utf-8")


print("Generator helpers ready.")


## 7. Generate Requirements and SDLC Documentation

**Purpose:** Generate or verify SDLC documentation required by the rubric.

**Output Files:** `requirements.md`, `prompt_design.md`, `architecture_decisions.md`, `safety_policy.md`, `assessment_traceability.md`


In [ ]:
doc_generation_prompts = {
    DOCS / "requirements.md": "Generate requirements.md with overview, objectives, functional and non-functional requirements, user stories, constraints, assumptions, API summary, database summary, out of scope, and future enhancements.",
    DOCS / "prompt_design.md": "Generate prompt_design.md explaining file-level generation prompts, runtime JSON prompts, validation rules, and common failure cases.",
    DOCS / "architecture_decisions.md": "Generate architecture_decisions.md explaining Flask, SQLite, APIFree, profile consent, safety rules, tests, Docker and deployment choices.",
    DOCS / "safety_policy.md": "Generate safety_policy.md for a non-diagnostic symptom-intake and triage-support website.",
    DOCS / "assessment_traceability.md": "Generate assessment_traceability.md mapping rubric requirements to files in generated_project.",
}

for path, prompt in doc_generation_prompts.items():
    fallback = existing_text(path, f"# {path.stem}\n\nGenerated documentation placeholder.\n")
    write_generated_file(path, BASE_PROMPT + "\n\n" + prompt, fallback)
    print("generated/verified:", path)


## 8. Generate UML

**Purpose:** Generate PlantUML source for software modelling and communication.

**Output Files:** `use_case.puml`, `class_diagram.puml`, `activity_diagram.puml`, broader full-site UML files, and `docs/uml_png/`


In [ ]:
uml_generation_prompts = {
    DOCS / "use_case.puml": "Generate a PlantUML use-case diagram for patient, developer, AI-service and clinician-review interactions in the AI symptom-intake website.",
    DOCS / "class_diagram.puml": "Generate a PlantUML class diagram for services, profiles, records, sessions, summaries, care guidance, and profile records.",
    DOCS / "activity_diagram.puml": "Generate a PlantUML activity diagram from symptom input through safety check, AI follow-up, summary, care guidance, and health-record structuring.",
    DOCS / "full_site_use_case.puml": "Generate a broader PlantUML use-case diagram for all pages and APIs.",
    DOCS / "full_site_class.puml": "Generate a broader PlantUML class/component model for the whole site.",
    DOCS / "full_site_activity.puml": "Generate a broader PlantUML activity flow for the whole site.",
    DOCS / "full_site_sequence.puml": "Generate a PlantUML sequence diagram showing Browser UI -> Flask backend -> Safety rules -> APIFree -> SQLite.",
}

for path, prompt in uml_generation_prompts.items():
    fallback = existing_text(path, "@startuml\n@enduml\n")
    write_generated_file(path, BASE_PROMPT + "\n\n" + prompt, fallback)
    print("generated/verified:", path)

print("Rendered UML PNG evidence folder:", UML_PNG)


## 9. Generate Database Schema

**Purpose:** Generate SQLite tables for profiles, health records, intake sessions, messages, summaries, and guidance.

**Output Files:** `schema.sql`


In [ ]:
schema_prompt = BASE_PROMPT + "\n\nGenerate schema.sql for the complete SQLite database. Return SQL only."
schema_fallback = existing_text(PROJECT / "schema.sql", "-- schema placeholder\n")
write_generated_file(PROJECT / "schema.sql", schema_prompt, schema_fallback)

print((PROJECT / "schema.sql").read_text(encoding="utf-8")[:900])


## 10. Generate Flask Backend

**Purpose:** Generate or verify the Flask backend, route layer, API endpoints, AI integration layer, safety rules, and fallback logic.

**Output Files:** `app.py`


In [ ]:
backend_prompt = BASE_PROMPT + dedent('''

Generate app.py for the complete Flask backend.
Include:
- app factory for tests
- website routes
- JSON API endpoints
- APIFree calls through Flask only
- red-flag safety rules
- profile/history consent
- safe non-diagnostic behaviour
Return Python only.
''').strip()

backend_fallback = existing_text(PROJECT / "app.py", "# app placeholder\n")
write_generated_file(PROJECT / "app.py", backend_prompt, backend_fallback)
print("Backend generated/verified:", PROJECT / "app.py")


## 11. Generate Frontend Pages and Generated Image

**Purpose:** Generate or verify website pages, CSS, JavaScript behaviour, and the required generated homepage image.

**Output Files:** `templates/*.html`, `static/styles.css`, `static/*.js`, `static/generated_banner.png`, `docs/generated_image_prompt.txt`


In [ ]:
frontend_files = sorted(list(TEMPLATES.glob("*.html")) + [
    STATIC / "styles.css",
    STATIC / "symptom_chat.js",
    STATIC / "profile.js",
    STATIC / "followup_keyboard.js",
])

for path in frontend_files:
    prompt = BASE_PROMPT + f"\n\nGenerate {path.name} for the AI symptom-intake website. Return only this file content."
    write_generated_file(path, prompt, existing_text(path, ""))
    print("generated/verified:", path)

image_prompt = dedent('''
Wide healthcare intake interface scene with symptom questionnaire cards, calm clinic setting,
privacy-conscious non-identifiable silhouettes, no diagnosis labels, no prescriptions, no logos,
no watermark.
''').strip()
(DOCS / "generated_image_prompt.txt").write_text(image_prompt + "\n", encoding="utf-8")

assert (STATIC / "generated_banner.png").exists(), "static/generated_banner.png is required"
assert "generated_banner.png" in (TEMPLATES / "index.html").read_text(encoding="utf-8")
print("Image prompt saved:", DOCS / "generated_image_prompt.txt")
print("Generated image verified:", STATIC / "generated_banner.png")


## 12. Generate Tests and Verify Outputs

**Purpose:** Generate or verify pytest coverage and prove the generated artefacts exist.

**Output Files:** `tests/conftest.py`, `tests/test_api.py`, `tests/test_routes.py`, verification output


In [ ]:
test_files = [TESTS / "conftest.py", TESTS / "test_api.py", TESTS / "test_routes.py"]

for path in test_files:
    prompt = BASE_PROMPT + f"

Generate pytest file {path.name} for this Flask project. Return Python only."
    write_generated_file(path, prompt, existing_text(path, ""))
    print("generated/verified:", path)

required_files = [
    PROJECT / "app.py",
    PROJECT / "requirements.txt",
    PROJECT / "schema.sql",
    PROJECT / "README.md",
    PROJECT / ".env.example",
    STATIC / "generated_banner.png",
    DOCS / "requirements.md",
    DOCS / "prompt_design.md",
    DOCS / "architecture_decisions.md",
    DOCS / "safety_policy.md",
    DOCS / "use_case.puml",
    DOCS / "class_diagram.puml",
    DOCS / "activity_diagram.puml",
    DOCS / "full_site_use_case.puml",
    DOCS / "full_site_activity.puml",
    DOCS / "full_site_sequence.puml",
    DOCS / "full_site_class.puml",
    UML_PNG / "use_case.png",
    UML_PNG / "activity_diagram.png",
    UML_PNG / "class_diagram.png",
    TESTS / "conftest.py",
    TESTS / "test_api.py",
    TESTS / "test_routes.py",
]

missing = [str(path) for path in required_files if not path.exists()]
assert not missing, "Missing required generated files: " + ", ".join(missing)

notebooks = list(Path(".").glob("*.ipynb"))
assert notebooks == [Path("notebook.ipynb")], f"Task 1 should contain exactly one notebook, found: {notebooks}"

unsafe_secret_files = [PROJECT / ".env"]
print(f"Verified {len(required_files)} required generated files.")
print("Task 1 notebook count verified:", notebooks)
print("Safe APIFree template verified:", PROJECT / ".env.example")
print("Local-only secret file allowed for development, but excluded from submission:", [p.name for p in unsafe_secret_files if p.exists()])
print("For Task 2, add screenshot evidence: commits.png, cicd.png, deployment.png")


## Optional Local Verification

Run these commands from `Task1/generated_project/` when checking the generated website locally:

```powershell
pip install -r requirements.txt
python app.py
```

Then open:

```text
http://127.0.0.1:5000/
http://127.0.0.1:5000/symptom-input?mode=quick
http://127.0.0.1:5000/api/health
```

For APIFree live AI output, create a local `.env` file in `Task1/generated_project/`:

```text
APIFREE_API_KEY=your_real_key
APIFREE_BASE_URL=https://api.apifree.ai/v1
APIFREE_MODEL=deepseek-ai/deepseek-v4-flash
FLASK_ENV=development
```

The Flask backend reads this file automatically with `python-dotenv`. Do not commit `.env`, API keys, local databases, caches, or logs.
